In [1]:
# Dataset analysis assignment
# Source: Corpus of Children's Video Media (CCVM), https://osf.io/grbyx/overview
# I want to explore how fast is the dialogue in kids TV shows, and how long is each line

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import collections

In [ ]:
# I have two datasets, so i set two path
episode_info_path = 'data/CCVM_Transcript-metadata.csv'
episode_info_df = pd.read_csv(episode_info_path, encoding='cp1252')

# "utt" is short for "utterance"
utt_path = 'data/CCVM_Utterance-length.csv'
utt_df = pd.read_csv(utt_path)

In [ ]:
episode_info_df.head()

In [ ]:
utt_df.head()

In [ ]:
print(episode_info_df.columns)
print(utt_df.columns)
print(len(episode_info_df), len(utt_df))

In [ ]:
# I will use this 3 element later: file_name, N_utterances, MLU(Mean Length of Utterance)


# Put the dialogue stats into episode_info_df
# file_name in utt_df ends with 'in.xml', remove it so it matches File_name in episode_info_df
# print(utt_df.file_name[0])

utt_df.file_name = utt_df.file_name.str.replace('in.xml', '')

# check the adjust file name
print(utt_df.file_name[0])

#########################################################################################################################

In [ ]:
# data preparation
n_utterances = []
mlu = []
for name in episode_info_df.File_name.tolist():
    n_utterances.append(utt_df.loc[utt_df.file_name == name, 'N_utterances'].item())
    mlu.append(utt_df.loc[utt_df.file_name == name, 'MLU'].item())

# add the two lists as new columns
episode_info_df['N_utterances'] = n_utterances
episode_info_df['MLU'] = mlu

# lines per minute
episode_info_df['Lines_per_min'] = episode_info_df.N_utterances / (episode_info_df.Length_seconds / 60)
episode_info_df[['Programme', 'Length_seconds', 'N_utterances', 'Lines_per_min', 'MLU']].head()


In [ ]:
# find the programme with the highest number of episode in the data
programmes = episode_info_df.Programme.tolist()
collections.Counter(programmes).most_common(5)

# most_common() is a method belonging to the Counter class within the built-in collections module. 
# It returns a list of (element, count) tuples sorted in descending order by their frequency

In [ ]:
episode_info_df.Style.value_counts()

In [ ]:
# find the dialogue pace information

In [ ]:
episode_info_df.Lines_per_min.describe()

#finding: kids TV episodes average about 17 lines per minute.

In [ ]:
plt.hist(episode_info_df.Lines_per_min, bins=20, color='pink')
plt.xlabel('Lines per minute')
plt.ylabel('Number of episodes')
plt.title('Dialogue pace in kids TV episodes')

# draw where the average is in blue
plt.axvline(episode_info_df.Lines_per_min.mean(), color='blue', linestyle='dashed', linewidth=2)

# draw where the median is in yellow
plt.axvline(episode_info_df.Lines_per_min.median(), color='yellow', linestyle='dashed', linewidth=2)

In [ ]:
# fastest and slowest episodes
fastest = episode_info_df.loc[episode_info_df.Lines_per_min == episode_info_df.Lines_per_min.max(), 'Programme'].item()
slowest = episode_info_df.loc[episode_info_df.Lines_per_min == episode_info_df.Lines_per_min.min(), 'Programme'].item()
fastest_per_min_max = episode_info_df.Lines_per_min.max()
slowest_per_min_min = episode_info_df.Lines_per_min.min()

print('Fastest:', fastest, fastest_per_min_max)
print('Slowest:', slowest, slowest_per_min_min)

In [ ]:
# Cartoon vs Live Action Show
# just want to find wether the cartoon and live action show difference

In [ ]:
style_groupings = episode_info_df.groupby('Style')

style_groupings.Lines_per_min.describe()

# Finding: cartoon and live-action have almost the same dialogue pace.

In [ ]:
cartoon_df = style_groupings.get_group('Cartoon')
live_action_df = style_groupings.get_group('Live-action')
plt.hist(cartoon_df.Lines_per_min, bins=20, alpha=0.5, label='Cartoon') #alpha: could tweak the bar graph alpha
plt.hist(live_action_df.Lines_per_min, bins=20, alpha=0.5, label='Live-action')
plt.legend()
plt.title('Lines per minute by style')
plt.xlabel('Lines per minute')
plt.ylabel('Number of episodes')

# Finding: cartoon and live-action shows center on the same range, about 15 to 20 lines per minute. 
# Cartoon bars are taller only because there are more cartoon episodes. 

In [ ]:
# Line Length (MLU)
episode_info_df.MLU.describe()

# Finding: lines in kids TV are short, about 5 words per line on average.

In [ ]:
plt.hist(episode_info_df.MLU, bins=20)
plt.xlabel('Words per line (MLU)')
plt.ylabel('Number of episodes')
plt.title('Line length in kids TV episodes')

# mean
plt.axvline(episode_info_df.MLU.mean(), color='yellow', linestyle='dashed', linewidth=2)

# median
plt.axvline(episode_info_df.MLU.median(), color='red', linestyle='dashed', linewidth=2)

In [ ]:
#Faster dialogue, shorter lines

In [ ]:
pacing_values = episode_info_df[['Length_seconds', 'N_utterances', 'Lines_per_min', 'MLU']]
pacing_values.corr()

In [ ]:
plt.matshow(pacing_values.corr())
plt.xticks(range(pacing_values.select_dtypes(['number']).shape[1]), pacing_values.select_dtypes(['number']).columns, fontsize=14, rotation=45)
plt.yticks(range(pacing_values.select_dtypes(['number']).shape[1]), pacing_values.select_dtypes(['number']).columns, fontsize=14)
cb = plt.colorbar()
cb.ax.tick_params(labelsize=14)
plt.title('Correlation Matrix', fontsize=16);
plt.show()

In [ ]:
# Teaser Reference
# at the average pace, how many lines fit in a 30 second teaser?
episode_info_df.Lines_per_min.mean() * 30 / 60

In [ ]:
# Findings
# 1. Kids TV episodes average about 17 lines per minute. Most episodes fall between 15 and 20.
# 2. "Bluey" is the fastest (about 30 lines per minute). "Tee and Mo" is the slowest (about 4.5).
# 3. Cartoon and live-action shows have almost the same pace, they both about 17 lines per minute).
# 4. Faster shows tend to use slightly shorter lines, but the correlation is weak (-0.24).
# 5. At the average pace, a 30 second teaser fits about 8 lines of dialogue.